# SOXX 发行人份额数原始历史：来源限定检查

仅核对 iShares 官方 Data Download 原字节、Historical 表、原生 Qlib 日历及原 Stage1 资金日索引。候选问题是一级市场份额创设/赎回是否提供原39输入与已关闭的二级成交量路线以外的信息。本卡不拟合、不读取2023年后收益、不开户回测。当前下载只有一个2026-10-22历史版本，不能证明历史首次可用时点。

In [1]:
import torch, qlib, json, hashlib, numpy as np, pandas as pd
from pathlib import Path
from qlib.data import D
from qlib.workflow import R
from src.data.soxx_issuer_fund_download import read_historical
from src.data.minute_intake import atomic_json
root=Path.cwd(); card=root/'research/soxx_issuer_shares_source_20261022'
plan=json.loads((card/'plan.json').read_text())
def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
raw=root/plan['raw_path']; provider=root/plan['qlib_daily_provider']
assert digest(raw)==plan['raw_sha256'] and digest(root/plan['parser'])==plan['parser_sha256']
assert digest(provider/'qlib_store_manifest.json')==plan['qlib_daily_manifest_sha256']
frame, parser_audit=read_historical(raw)
qlib.init(provider_uri=str(provider),region='us',kernels=1)
full=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-10-09',freq='day'))
train=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2022-12-30',freq='day'))
issuer=frame.reindex(full)
assert not issuer.index.duplicated().any()
shares=issuer.shares_outstanding
# Source-only provisional shape: two Qlib sessions of publication lag and five-session change.
feature=np.log(shares.shift(2)/shares.shift(7))
source=plan['funded_date_source']
with R.uri_context(str(root/source['card']/'qlib_recorder')):
 recorder=R.get_recorder(recorder_id=source['recorder_id'],experiment_id=source['experiment_id'])
 funded=pd.DatetimeIndex(recorder.load_object(source['artifact']).index)
assert funded.is_monotonic_increasing and not funded.has_duplicates
split_prev=frame.loc['2024-03-06']; split_next=frame.loc['2024-03-07']
result={'status':'complete_retrospective_issuer_history_first_public_vintage_unqualified_no_fit',
 'plan_sha256':digest(card/'plan.json'),'raw_sha256':digest(raw),'raw_bytes':raw.stat().st_size,
 'source_response_date_header_utc':plan['response_date_header_utc'],
 'parser_audit':parser_audit,'history_first':str(frame.index.min().date()),
 'history_last':str(frame.index.max().date()),'weekend_rows':int((frame.index.dayofweek>=5).sum()),
 'missing_shares_calendar_dates':[str(value.date()) for value in frame.index[frame.shares_outstanding.isna()]],
 'qlib_train_sessions':len(train),'issuer_train_missing_sessions':int(issuer.reindex(train).shares_outstanding.isna().sum()),
 'provisional_lagged_feature_train_missing':int(feature.reindex(train).isna().sum()),
 'funded_signal_sessions':len(funded),'issuer_funded_missing_sessions':int(issuer.reindex(funded).shares_outstanding.isna().sum()),
 'provisional_lagged_feature_funded_missing':int(feature.reindex(funded).isna().sum()),
 'split_2024_03_07_share_ratio':float(split_next.shares_outstanding/split_prev.shares_outstanding),
 'split_2024_03_07_nav_ratio':float(split_next.nav_per_share/split_prev.nav_per_share),
 'historical_first_public_clock_qualified':False,'historical_revision_vintages_qualified':False,
 'shares_are_primary_market_issuer_measure_not_secondary_volume':True,
 'evaluation_outcome_values_loaded':0,'new_model_fits':0,'new_native_accounts':0,
 'next_action':'Obtain dated original vintages or issuer publication evidence for shares outstanding; only then freeze a matched original Stage1 information experiment. Do not use two-session lag as proof of historical point-in-time availability.'}
archive=json.loads((card/'nas_source_archive_receipt.json').read_text())
assert archive['source_sha256']==result['raw_sha256'] and archive['restored_original_bytes_equal']
assert archive['remote_sha256_readback']==archive['archive_sha256']
result['nas_archive_sha256']=archive['archive_sha256']
result['nas_scoped_restoration_verified']=True
atomic_json(result,card/'source_audit.json')
result


{'status': 'complete_retrospective_issuer_history_first_public_vintage_unqualified_no_fit',
 'plan_sha256': 'fc67868c6fff7b904c150bd86243abb129abf7070b3eb06aaab27800d2fbd5d9',
 'raw_sha256': '74265a678335a4fb5aac42cb854a60ecd3afc57b8d5ffd1005ed5ccadd5e07b7',
 'raw_bytes': 3347697,
 'source_response_date_header_utc': '2026-10-22 14:46:39 GMT',
 'parser_audit': {'schema': ['As Of',
   'NAV per Share',
   'Ex-Dividends',
   'Shares Outstanding'],
  'daily_calendar_rows': 8937,
  'historical_missing_nav_rows': 0,
  'historical_missing_shares_rows': 2,
  'invalid_xml_ampersands_escaped_in_memory': 6,
  'raw_source_modified': False},
 'history_first': '2001-07-10',
 'history_last': '2026-10-21',
 'weekend_rows': 2365,
 'missing_shares_calendar_dates': ['2024-03-29', '2024-03-31'],
 'qlib_train_sessions': 3020,
 'issuer_train_missing_sessions': 0,
 'provisional_lagged_feature_train_missing': 0,
 'funded_signal_sessions': 944,
 'issuer_funded_missing_sessions': 0,
 'provisional_lagged_feature_

In [2]:
from bs4 import BeautifulSoup
sample_plan=json.loads((card/'vintage_sample_plan.json').read_text())
vintage_archive=json.loads((card/'nas_vintage_archive_receipt.json').read_text())
assert vintage_archive['scoped_restoration_verified']
observations=[]
for sample in sample_plan['samples']:
 path=root/'data/raw/external/soxx_issuer_shares_20261022/vintages'/(sample['label']+'.html')
 item=vintage_archive['source_objects'][str(path.relative_to(root))]
 assert digest(path)==item['sha256'] and path.stat().st_size==item['bytes']
 page=BeautifulSoup(path.read_bytes(),'html.parser')
 node=page.select_one('tr.sharesOutstanding, div.col-sharesOutstanding')
 row={'capture_label':sample['label'],'archive_timestamp_utc':sample['wayback_timestamp'],
      'original_url':sample['replay_url'],'raw_sha256':digest(path),'raw_bytes':path.stat().st_size}
 if node:
  asof=node.select_one('.as-of-date');value=node.select_one('.data')
  if asof and value:
   as_of=pd.Timestamp(asof.get_text(' ',strip=True).removeprefix('as of ').strip())
   published=int(value.get_text(' ',strip=True).replace(',',''))
   current=int(frame.loc[as_of,'shares_outstanding'])
   row.update(status='issuer_page_share_value_extracted',issuer_as_of=str(as_of.date()),
              archived_page_shares=published,current_workbook_split_adjusted_shares=current,
              workbook_to_page_ratio=current/published)
  else:row['status']='shares_field_container_incomplete'
 else:row['status']='shares_field_not_extractable_from_capture'
 observations.append(row)
vintage={'status':'three_original_page_values_match_known_2024_split_units_fourth_not_extractable',
 'source_archive_sha256':vintage_archive['archive_sha256'],'observations':observations,
 'match_count_exact_3x':sum(o.get('workbook_to_page_ratio')==3 for o in observations),
 'known_2024_split_primary_source':'https://www.ishares.com/us/literature/tax-information/form-8937-ishares-semiconductor-etf-3-6-2024.pdf',
 'limitation':'Sparse archive captures show selected historical issuer shares and current workbook restatement. They do not prove full 2011-2022 daily first-public timing, complete revision history or live receipt before every Stage1 decision.',
 'first_public_clock_qualified':False,'revision_vintages_qualified':False,
 'new_model_fits':0,'new_native_accounts':0,'evaluation_outcome_values_loaded':0}
atomic_json(vintage,card/'vintage_audit.json')
vintage


{'status': 'three_original_page_values_match_known_2024_split_units_fourth_not_extractable',
 'source_archive_sha256': '062ce54864a40f6f7130067a6e1ee152dff5cd721d553d3189a6d0373daec183',
 'observations': [{'capture_label': '2016-01-28',
   'archive_timestamp_utc': '20160128030203',
   'original_url': 'https://web.archive.org/web/20160128030203id_/https://www.ishares.com/us/products/239705/ishares-phlx-semiconductor-etf',
   'raw_sha256': 'a06fa1ff68510b5e769f9a899c18a3737069a4dc6c63eda0788fb1c296d4f61b',
   'raw_bytes': 541125,
   'status': 'issuer_page_share_value_extracted',
   'issuer_as_of': '2016-01-27',
   'archived_page_shares': 3900000,
   'current_workbook_split_adjusted_shares': 11700000,
   'workbook_to_page_ratio': 3.0},
  {'capture_label': '2018-01-02',
   'archive_timestamp_utc': '20180102014316',
   'original_url': 'https://web.archive.org/web/20180102014316id_/https://www.ishares.com/us/products/239705/ishares-phlx-semiconductor-etf',
   'raw_sha256': 'dbce5589ce408b878

In [3]:
training_source=plan['training_redundancy_source']
artifact_path=(root/training_source['card']/'qlib_recorder'/training_source['experiment_id']
               /training_source['recorder_id']/'artifacts'/training_source['artifact'])
assert digest(artifact_path)==training_source['artifact_sha256']
with R.uri_context(str(root/training_source['card']/'qlib_recorder')):
 source_recorder=R.get_recorder(recorder_id=training_source['recorder_id'],
                               experiment_id=training_source['experiment_id'])
 original_features=source_recorder.load_object(training_source['artifact'])
original_names=list(original_features.columns[:39])
assert len(original_names)==39 and not any('shares_outstanding' in name for name in original_names)
paired=pd.DataFrame({'issuer_share_change_lagged_5':feature.reindex(train)})
paired=paired.join(original_features.reindex(train)[original_names])
assert paired.issuer_share_change_lagged_5.notna().sum()==3020
correlations={name:float(paired['issuer_share_change_lagged_5'].corr(paired[name],method='spearman'))
              for name in original_names}
assert all(np.isfinite(value) for value in correlations.values())
sorted_names=sorted(correlations,key=lambda name:abs(correlations[name]),reverse=True)
redundancy={'status':'training_only_retrospective_redundancy_diagnostic_not_information_gate',
 'plan_sha256':digest(card/'plan.json'),'source_recorder_id':training_source['recorder_id'],
 'source_artifact_sha256':training_source['artifact_sha256'],'training_rows':len(paired),
 'feature_names_original_39':original_names,'spearman_against_original_39':correlations,
 'maximum_absolute_spearman':abs(correlations[sorted_names[0]]),
 'top_five_absolute_spearman':[{'feature':name,'spearman':correlations[name]} for name in sorted_names[:5]],
 'count_absolute_spearman_ge_0_8':sum(abs(value)>=0.8 for value in correlations.values()),
 'historical_first_public_clock_qualified':False,'new_model_fits':0,'new_native_accounts':0,
 'evaluation_outcome_values_loaded':0,
 'limitation':'Correlation on the original training inputs is not incremental forecast information or alpha.'}
atomic_json(redundancy,card/'redundancy_audit.json')
redundancy['top_five_absolute_spearman']


[{'feature': 'soxx_ret10', 'spearman': 0.12974865049462278},
 {'feature': 'soxl_ret10', 'spearman': 0.12973816508968097},
 {'feature': 'semis_vs_cross_20', 'spearman': 0.1100672628795163},
 {'feature': 'soxx_trend60', 'spearman': 0.10696207803058003},
 {'feature': 'range_pos_252', 'spearman': 0.10277802715366802}]

## 中文结论

官方文件覆盖2011–2022全部3020个Qlib训练交易日和944个已封存资金信号日，原文件与网页存档原字节已NAS冷回读。三个可提取的历史发行人页面份额数与当前文件按2024年3拆1单位换算后精确相符；第四个2022页面未提取到份额字段。训练期与原39特征的冗余关系见 redundancy_audit.json，但相关性不是收益信息。稀疏网页存档不足以认证全部历史日的首次公布和修订。0拟合、0账户、0评价收益读取。下一步补齐时点证据后，才冻结匹配原Stage1的信息检验。